# 08 — Adaptive White-Box Evaluation (Skenario Terburuk)

**Menutup gap #3 Paper 1:** evaluasi sebelumnya (T5) memakai serangan *transfer/static* — penyerang tak tahu pertahanan. Skenario dunia-nyata terburuk: penyerang **tahu persis model pertahanan** dan membangkitkan evasion **langsung dari gradien model pertahanan itu sendiri** (*adaptive white-box*).

**Pertanyaan inti:** apakah model *robust* (adversarial-trained, T5) yang tampak kuat di bawah serangan transfer **tetap kuat** di bawah serangan adaptive? Jika MCC-nya jatuh lagi → pertahanan hanya efektif thd transfer, bukan adaptive (temuan jujur, menutup gap #3). Jika bertahan → pertahanan sejati.

**Desain (Model A, biner, 2 arah CIC & UNSW):**
- Dua model per arah: **baseline** (tanpa adv-training) & **robust** (adv-training, D_clean ∪ D_adv 80:20, eps_train=0.1).
- Dua serangan:
  - **Transfer/static** — saliency dihitung dari model **baseline**, diterapkan ke target.
  - **Adaptive white-box** — saliency dihitung dari **model yang diserang itu sendiri**.
- Semua serangan memakai **functional-preserving constraint** (T8) agar realistis (flow valid).

**Matriks evaluasi (per arah, eps grid):**
| Model diserang | Serangan | Makna |
|---|---|---|
| baseline | adaptive-thd-baseline | kerentanan model tak-bertahan (batas bawah) |
| robust | transfer-dari-baseline | ketahanan thd serangan statis (klaim T5) |
| robust | adaptive-thd-robust | ketahanan thd penyerang menyesuaikan (skenario terburuk) |

**Hipotesis jujur:** literatur menunjukkan adv-training sering runtuh di bawah adaptive white-box (obfuscated gradients). Sangat mungkin MCC robust turun lagi saat adaptive. Itu temuan berharga — dilaporkan apa adanya.

> Jalankan di SageMaker.

In [ ]:
# --- Bootstrap ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost')]:
    try: importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg}'); subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import pickle, os, json
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
OUT_JSON='../adaptive_whitebox.json'
SEED=42; H=0.01; EPS_EVAL=[0.05,0.1,0.2]; EPS_TRAIN=0.1; ADV_RATIO=0.20; MAXN=40000
print('files:', os.path.exists(CIC_PKL), os.path.exists(UNSW_TRAIN), os.path.exists(UNSW_TEST))

In [ ]:
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys()); IX={c:i for i,c in enumerate(CANON)}

def build_matrix(df, side):
    idx=0 if side=='cic' else 1
    cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON
    out=out.replace([np.inf,-np.inf],np.nan)
    out=out.fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float).values

def make_xgb():
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=SEED,tree_method='hist')

def ev(yt,yp):
    return dict(mcc=float(matthews_corrcoef(yt,yp)),f1=float(f1_score(yt,yp,zero_division=0)),
                acc=float(accuracy_score(yt,yp)),confusion=confusion_matrix(yt,yp).tolist())

In [ ]:
# --- Serangan: saliency (finite-diff) + FGSM + functional-preserving projection ---
def loss_bin(model,X,y):
    p=np.clip(model.predict_proba(X)[:,1],1e-15,1-1e-15); y=y.astype(float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))

def saliency(model,X,y,h=H):
    n,m=X.shape; S=np.zeros((n,m))
    for i in range(m):
        Xp=X.copy(); Xp[:,i]+=h; Xm=X.copy(); Xm[:,i]-=h
        S[:,i]=(loss_bin(model,Xp,y)-loss_bin(model,Xm,y))/(2*h)
    return S

def project_functional(Xadv_scaled, scaler):
    Xo=Xadv_scaled*scaler.scale_+scaler.mean_; Xo=Xo.copy()
    Xo=np.clip(Xo,0.0,None)
    Xo[:,IX['fwd_pkts']]=np.round(Xo[:,IX['fwd_pkts']]); Xo[:,IX['bwd_pkts']]=np.round(Xo[:,IX['bwd_pkts']])
    Xo[:,IX['fwd_bytes']]=np.maximum(Xo[:,IX['fwd_bytes']],Xo[:,IX['fwd_pkts']])
    Xo[:,IX['bwd_bytes']]=np.maximum(Xo[:,IX['bwd_bytes']],Xo[:,IX['bwd_pkts']])
    with np.errstate(divide='ignore',invalid='ignore'):
        fm=np.where(Xo[:,IX['fwd_pkts']]>0,Xo[:,IX['fwd_bytes']]/Xo[:,IX['fwd_pkts']],0.0)
        bm=np.where(Xo[:,IX['bwd_pkts']]>0,Xo[:,IX['bwd_bytes']]/Xo[:,IX['bwd_pkts']],0.0)
    Xo[:,IX['fwd_mean']]=fm; Xo[:,IX['bwd_mean']]=bm
    return (Xo-scaler.mean_)/scaler.scale_

def fgsm_functional(X_scaled, S, eps, scaler):
    Xadv=X_scaled+eps*np.sign(S)
    Xo_adv=Xadv*scaler.scale_+scaler.mean_; Xo_ref=X_scaled*scaler.scale_+scaler.mean_
    for j in [IX[c] for c in ['fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','duration']]:
        Xo_adv[:,j]=np.maximum(Xo_adv[:,j],Xo_ref[:,j])   # monotonic add-only
    Xadv=(Xo_adv-scaler.mean_)/scaler.scale_
    return project_functional(Xadv, scaler)

In [ ]:
# --- Muat data + z-score per dataset ---
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float)
sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0)
y_cic=(np.asarray(d['y'])!=benign).astype(int)
unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values

Xc_all=build_matrix(cic_df,'cic')
Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=SEED,stratify=y_cic)
scc=StandardScaler().fit(Xc_tr_raw); Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
scu=StandardScaler().fit(Xu_tr_raw); Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
print('CIC tr/te:',Xc_tr.shape,Xc_te.shape,'| UNSW tr/te:',Xu_tr.shape,Xu_te.shape)

In [ ]:
# --- Latih baseline + robust (adversarial training) per arah ---
def train_pair(X_tr, y_tr, scaler, tag):
    base=make_xgb(); base.fit(X_tr,y_tr)
    # adversarial training: adv dibangkitkan thd baseline (functional), augmentasi 80:20
    rng=np.random.RandomState(SEED)
    n=min(MAXN,len(X_tr)); idx=rng.choice(len(X_tr),n,replace=False)
    S=saliency(base,X_tr[idx],y_tr[idx])
    Xadv=fgsm_functional(X_tr[idx],S,EPS_TRAIN,scaler)
    n_adv=int(len(X_tr)*ADV_RATIO/(1-ADV_RATIO)); n_adv=min(n_adv,len(Xadv))
    sel=rng.choice(len(Xadv),n_adv,replace=False)
    Xrob=np.vstack([X_tr,Xadv[sel]]); yrob=np.concatenate([y_tr,y_tr[idx][sel]])
    rob=make_xgb(); rob.fit(Xrob,yrob)
    print(f'[{tag}] baseline+robust dilatih (D_robust={len(Xrob):,})')
    return base, rob

m_cic_base, m_cic_rob = train_pair(Xc_tr, yc_tr, scc, 'CIC')
m_unsw_base, m_unsw_rob = train_pair(Xu_tr, y_utr, scu, 'UNSW')

In [ ]:
# --- Evaluasi transfer vs adaptive (functional) ---
def evaluate_whitebox(base, rob, X, y, scaler, tag):
    rng=np.random.RandomState(SEED)
    n=min(MAXN,len(X)); idx=rng.choice(len(X),n,replace=False)
    Xs,ys=X[idx],y[idx]
    # saliency thd baseline (utk transfer) & thd robust (utk adaptive)
    S_base=saliency(base,Xs,ys)
    S_rob =saliency(rob, Xs,ys)
    out={'baseline_clean':ev(ys,base.predict(Xs)), 'robust_clean':ev(ys,rob.predict(Xs))}
    for e in EPS_EVAL:
        # serangan functional dari masing-masing sumber gradien
        Xadv_from_base=fgsm_functional(Xs,S_base,e,scaler)
        Xadv_from_rob =fgsm_functional(Xs,S_rob, e,scaler)
        # baseline diserang adaptif thd dirinya
        out[f'baseline_adaptive_eps{e}']=ev(ys, base.predict(Xadv_from_base))
        # robust: transfer (gradien baseline) vs adaptive (gradien robust)
        out[f'robust_transfer_eps{e}']  =ev(ys, rob.predict(Xadv_from_base))
        out[f'robust_adaptive_eps{e}']  =ev(ys, rob.predict(Xadv_from_rob))
    print(f'\n[{tag}] clean: baseline={out["baseline_clean"]["mcc"]:.4f} robust={out["robust_clean"]["mcc"]:.4f}')
    for e in EPS_EVAL:
        print(f'  eps={e}: base(adaptive)={out[f"baseline_adaptive_eps{e}"]["mcc"]:+.4f}'
              f' | robust(transfer)={out[f"robust_transfer_eps{e}"]["mcc"]:+.4f}'
              f' | robust(adaptive)={out[f"robust_adaptive_eps{e}"]["mcc"]:+.4f}')
    return out

results={}
print('='*72); print('ARAH CIC (uji CIC test)'); print('='*72)
results['cic']=evaluate_whitebox(m_cic_base, m_cic_rob, Xc_te, yc_te, scc, 'CIC')
print('\n'+'='*72); print('ARAH UNSW (uji UNSW test)'); print('='*72)
results['unsw']=evaluate_whitebox(m_unsw_base, m_unsw_rob, Xu_te, y_ute, scu, 'UNSW')

In [ ]:
# --- Ringkasan kunci (eps=0.1): apakah robust bertahan thd adaptive? ---
print('RINGKASAN (eps=0.1) — MCC:')
print(f"{'arah':<8}{'robust_clean':>14}{'robust_transfer':>17}{'robust_adaptive':>17}")
rows=[]
for a in ['cic','unsw']:
    r=results[a]
    rc=r['robust_clean']['mcc']; rt=r['robust_transfer_eps0.1']['mcc']; ra=r['robust_adaptive_eps0.1']['mcc']
    print(f"{a:<8}{rc:>14.4f}{rt:>17.4f}{ra:>17.4f}")
    rows.append(dict(arah=a, robust_clean=round(rc,4), robust_transfer=round(rt,4),
                     robust_adaptive=round(ra,4), drop_adaptive=round(rt-ra,4)))
print('\nInterpretasi: jika robust_adaptive << robust_transfer -> pertahanan runtuh thd penyerang adaptif')
print('(hanya efektif thd transfer/static). Jika robust_adaptive ~ robust_transfer -> pertahanan sejati.')

meta=dict(
  deskripsi='Adaptive white-box evaluation (functional-preserving). Model A biner, 2 arah.',
  features=CANON, eps_eval=EPS_EVAL,
  config=dict(eps_train=EPS_TRAIN, adv_ratio=ADV_RATIO, constraints='functional-preserving (T8)'),
  summary_eps01=rows, results=results,
)
with open(OUT_JSON,'w') as f: json.dump(meta,f,indent=2)
print('\nSaved:', OUT_JSON)